# Unified evaluation — Biomed GT2KG / GCN / Sentence-BERT

Protocole complet : 4 dimensions cachées × 10 splits × 5 seeds d'entraînement = **200 runs**. Le hidden est sélectionné uniquement sur la moyenne validation.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import pandas as pd

from tdg_bench import TDGBench

PROJECT_ROOT = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "pyproject.toml").is_file()
)
bench = TDGBench(PROJECT_ROOT / "benchmark_data")

In [ ]:
result = bench.unified_evaluation(
    dataset="biomed",
    graph="GT2KG_tdg",
    model="GCN",
    initial_embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    hidden_dims=[64, 128, 256, 512],
    split_seeds=None,          # les 10 splits disponibles
    training_seeds=[1, 2, 3, 4, 5],
    feature_seed=42,           # utile seulement pour la baseline random
    metrics="all",
    selection_metric="f1_macro",
    class_wise=True,
    max_epochs=100,
    patience=100,
    persist=True,             # sauvegarde chaque run terminé
    resume=True,              # reprend automatiquement après un arrêt
    device="auto",
    verbose=True,              # progression des 200 runs
    epoch_verbose=False,       # évite 200 journaux par epoch
)

In [ ]:
print("Dossier des résultats :", result.results_dir)
print("Runs repris :", result.resumed_runs)
print("Runs entraînés maintenant :", result.completed_runs)

rows = []
for hidden, summary in result.by_hidden_dim.items():
    validation = summary.validation[result.selection_metric]
    test = summary.test[result.selection_metric]
    rows.append({
        "hidden_dim": hidden,
        "validation_mean": validation.mean,
        "validation_std_split": validation.std_split,
        "validation_std_seed": validation.std_seed,
        "test_mean": test.mean,
        "test_std_split": test.std_split,
        "test_std_seed": test.std_seed,
        "test_std_total": test.std_total,
    })

hidden_summary = pd.DataFrame(rows).set_index("hidden_dim")
display(hidden_summary)
print("Hidden sélectionné :", result.selected_hidden_dim)

In [ ]:
selected = result.selected_summary
selected_test = pd.DataFrame({
    metric: {
        "mean": values.mean,
        "std_split": values.std_split,
        "std_seed": values.std_seed,
        "std_total": values.std_total,
    }
    for metric, values in selected.test.items()
}).T
display(selected_test)

In [ ]:
class_rows = []
for label, metrics in selected.class_wise["test"].items():
    row = {"label": label}
    for metric in ("precision", "recall", "f1"):
        row[f"{metric}_mean"] = metrics[metric].mean
        row[f"{metric}_std_split"] = metrics[metric].std_split
        row[f"{metric}_std_seed"] = metrics[metric].std_seed
    class_rows.append(row)

display(pd.DataFrame(class_rows).set_index("label"))